In [ ]:
#@title Marketing Campaign Optimization Dashboard (Enhanced)
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import io
import warnings
import base64
import datetime
warnings.filterwarnings('ignore')

# Initialize global variables
df = None
uploaded_file = None

# Create a function to open the dashboard in a new window


def open_dashboard():
    display(Javascript('''
        window.open(
            "https://colab.research.google.com/drive/1MgDI5T7c7DNr_nwqRUww9q1w-6odhnaV",
            "_blank"
        );
    '''))

open_dashboard()
# Create widgets
upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV'
)

process_button = widgets.Button(
    description='Analyze Data',
    disabled=False,
    button_style='success',
    tooltip='Click to analyze',
    layout=widgets.Layout(width='150px', height='40px')
)

open_window_button = widgets.Button(
    description='Open in New Window',
    disabled=False,
    button_style='info',
    tooltip='Open dashboard in new window',
    layout=widgets.Layout(width='150px', height='40px')
)

output = widgets.Output()

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h1 style='color: #2b5876;'>Marketing Campaign Optimization Dashboard</h1>"),
    widgets.HTML("<h3 style='color: #4e4376;'>Upload your marketing campaign dataset (CSV format)</h3>"),
    widgets.HBox([upload, process_button, open_window_button]),
    output
]))

# Function to clean and preprocess data
def clean_data(df):
    """Clean and preprocess the dataframe"""
    # Strip any leading/trailing spaces in column names
    df.columns = df.columns.str.strip()

    # Convert Campaign_ID to string if exists
    if 'Campaign_ID' in df.columns:
        df['Campaign_ID'] = df['Campaign_ID'].astype(str)

    # Convert Acquisition_Cost to float if it contains $ and ,
    if 'Acquisition_Cost' in df.columns:
        if df['Acquisition_Cost'].dtype == 'object':
            df['Acquisition_Cost'] = df['Acquisition_Cost'].astype(str)
            df['Acquisition_Cost'] = df['Acquisition_Cost'].str.replace('$', '', regex=False)
            df['Acquisition_Cost'] = df['Acquisition_Cost'].str.replace(',', '', regex=False)
            df['Acquisition_Cost'] = df['Acquisition_Cost'].astype(float)

    # Convert Date to datetime if exists
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

    return df

# All visualization functions with descriptions
def plot_conversion_by_campaign_type(df):
    """Plot average conversion rate by campaign type"""
    if 'Campaign_Type' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n1. Average Conversion Rate by Campaign Type")
        print("This visualization shows which campaign types generate the highest conversion rates.")
        print("Focus your budget on high-performing types and investigate why others underperform.")

    avg_data = df.groupby(['Campaign_Type'])['Conversion_Rate'].mean()
    colors = ['#FC4180', '#6056B3', '#FFC554', '#90CA78', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Campaign Type', pad=20, fontsize=14)
    plt.xlabel('Campaign Types', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_roi_by_campaign_type(df):
    """Plot average ROI by campaign type"""
    if 'Campaign_Type' not in df.columns or 'ROI' not in df.columns:
        return None

    with output:
        print("\n2. Average ROI by Campaign Type")
        print("This shows the return on investment for each campaign type.")
        print("Prioritize types with highest ROI for maximum profitability.")

    avg_data = df.groupby(['Campaign_Type'])['ROI'].mean()
    colors = ['#FCA180', '#805683', '#FFC55A', '#98C4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average ROI by Campaign Type', pad=20, fontsize=14)
    plt.xlabel('Campaign Type', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average ROI', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_acquisition_cost_by_channel(df):
    """Plot average acquisition cost by channel and campaign type"""
    if 'Channel_Used' not in df.columns or 'Campaign_Type' not in df.columns or 'Acquisition_Cost' not in df.columns:
        return None

    with output:
        print("\n3. Average Acquisition Cost by Channel and Campaign Type")
        print("This heatmap shows where you're spending the most to acquire customers.")
        print("Look for cost-effective combinations to optimize your marketing spend.")

    avg_data = df.groupby(['Channel_Used', 'Campaign_Type'])['Acquisition_Cost'].mean().unstack()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(14, 7))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Acquisition Cost by Channel and Campaign Type', pad=20, fontsize=14)
    plt.xlabel('Channel', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Acquisition Cost ($)', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'${height:,.0f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, -30), textcoords='offset points',
                     rotation=90)

    plt.tight_layout()
    plt.show()

def plot_conversion_heatmap(df):
    """Plot conversion rate heatmap by audience, type, and channel"""
    if not all(col in df.columns for col in ['Target_Audience', 'Campaign_Type', 'Channel_Used', 'Conversion_Rate']):
        return None

    with output:
        print("\n4. Conversion Rate Heatmap by Audience, Type, and Channel")
        print("This detailed heatmap reveals high-performing combinations.")
        print("Use it to identify your most effective targeting strategies.")

    avg_data = df.groupby(['Target_Audience', 'Campaign_Type', 'Channel_Used'])['Conversion_Rate'].mean().unstack()

    plt.figure(figsize=(16, 8))
    sns.heatmap(avg_data, annot=True, fmt='.2%', linewidths=0.5, cmap='YlGnBu')
    plt.title('Average Conversion Rate by Target Audience, Campaign Type, and Channel', pad=20, fontsize=14)
    plt.xlabel('Channel', labelpad=10)
    plt.ylabel('Target Audience and Campaign Type', labelpad=10)
    plt.tight_layout()
    plt.show()

def plot_engagement_by_segment(df):
    """Plot average engagement score by customer segment"""
    if 'Customer_Segment' not in df.columns or 'Engagement_Score' not in df.columns:
        return None

    with output:
        print("\n5. Average Engagement Score by Customer Segment")
        print("This shows which customer groups are most engaged with your campaigns.")
        print("Highly engaged segments may respond well to upsell opportunities.")

    avg_data = df.groupby('Customer_Segment')['Engagement_Score'].mean()
    colors = ['#FC4180', '#6056B3', '#FFC554', '#90CA78', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Engagement Score by Customer Segment', pad=20, fontsize=14)
    plt.xlabel('Customer Segment', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Engagement Score', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_conversion_by_segment(df):
    """Plot average conversion rate by customer segment"""
    if 'Customer_Segment' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n6. Average Conversion Rate by Customer Segment")
        print("Identifies which customer segments convert at the highest rates.")
        print("Consider focusing your targeting on these high-converting groups.")

    avg_data = df.groupby('Customer_Segment')['Conversion_Rate'].mean()
    colors = ['#FCA180', '#805683', '#FFC55A', '#98C4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Customer Segment', pad=20, fontsize=14)
    plt.xlabel('Customer Segment', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_engagement_by_channel(df):
    """Plot average engagement score by channel"""
    if 'Channel_Used' not in df.columns or 'Engagement_Score' not in df.columns:
        return None

    with output:
        print("\n7. Average Engagement Score by Channel")
        print("Shows which channels generate the most engagement.")
        print("High engagement often correlates with brand loyalty and repeat purchases.")

    avg_data = df.groupby('Channel_Used')['Engagement_Score'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Engagement Score by Channel', pad=20, fontsize=14)
    plt.xlabel('Channel', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Engagement Score', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_conversion_by_channel(df):
    """Plot average conversion rate by channel"""
    if 'Channel_Used' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n8. Average Conversion Rate by Channel")
        print("Identifies the most effective channels for driving conversions.")
        print("Allocate more budget to high-converting channels.")

    avg_data = df.groupby('Channel_Used')['Conversion_Rate'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Channel', pad=20, fontsize=14)
    plt.xlabel('Channel', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_roi_by_channel(df):
    """Plot average ROI by channel"""
    if 'Channel_Used' not in df.columns or 'ROI' not in df.columns:
        return None

    with output:
        print("\n9. Average ROI by Channel")
        print("Shows which channels deliver the best return on investment.")
        print("Prioritize channels with highest ROI for maximum profitability.")

    avg_data = df.groupby('Channel_Used')['ROI'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average ROI by Channel', pad=20, fontsize=14)
    plt.xlabel('Channel', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average ROI', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_engagement_by_duration(df):
    """Plot average engagement score by duration"""
    if 'Duration' not in df.columns or 'Engagement_Score' not in df.columns:
        return None

    with output:
        print("\n10. Average Engagement Score by Duration")
        print("Reveals how campaign duration affects engagement.")
        print("Use this to optimize the length of your campaigns.")

    avg_data = df.groupby('Duration')['Engagement_Score'].mean()
    colors = ['#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Engagement Score by Duration', pad=20, fontsize=14)
    plt.xlabel('Duration', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Engagement Score', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_conversion_by_duration(df):
    """Plot average conversion rate by duration"""
    if 'Duration' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n11. Average Conversion Rate by Duration")
        print("Shows how campaign length affects conversion rates.")
        print("Find the sweet spot for duration that maximizes conversions.")

    avg_data = df.groupby('Duration')['Conversion_Rate'].mean()
    colors = ['#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Duration', pad=20, fontsize=14)
    plt.xlabel('Duration', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_roi_by_duration(df):
    """Plot average ROI by duration"""
    if 'Duration' not in df.columns or 'ROI' not in df.columns:
        return None

    with output:
        print("\n12. Average ROI by Duration")
        print("Identifies which campaign durations yield the best returns.")
        print("Optimize campaign length based on ROI, not just conversions.")

    avg_data = df.groupby('Duration')['ROI'].mean()
    colors = ['#FFC55A', '#9BC4F8', '#59D5E0', '#FB773C']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average ROI by Duration', pad=20, fontsize=14)
    plt.xlabel('Duration', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average ROI', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_engagement_by_location(df):
    """Plot average engagement score by location"""
    if 'Location' not in df.columns or 'Engagement_Score' not in df.columns:
        return None

    with output:
        print("\n13. Average Engagement Score by Location")
        print("Shows geographic variations in engagement.")
        print("Use this to tailor content by region or identify expansion opportunities.")

    avg_data = df.groupby('Location')['Engagement_Score'].mean()
    colors = ['#FC4180', '#6056B3', '#FFC554', '#90CA78', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Engagement Score by Location', pad=20, fontsize=14)
    plt.xlabel('Location', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Engagement Score', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_conversion_by_location(df):
    """Plot average conversion rate by location"""
    if 'Location' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n14. Average Conversion Rate by Location")
        print("Identifies high-performing geographic markets.")
        print("Consider regional targeting strategies based on these insights.")

    avg_data = df.groupby('Location')['Conversion_Rate'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Location', pad=20, fontsize=14)
    plt.xlabel('Location', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_roi_by_location(df):
    """Plot average ROI by location"""
    if 'Location' not in df.columns or 'ROI' not in df.columns:
        return None

    with output:
        print("\n15. Average ROI by Location")
        print("Shows which geographic markets deliver the best returns.")
        print("Allocate budget to high-ROI locations and investigate underperformers.")

    avg_data = df.groupby('Location')['ROI'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average ROI by Location', pad=20, fontsize=14)
    plt.xlabel('Location', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average ROI', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_engagement_by_language(df):
    """Plot average engagement score by language"""
    if 'Language' not in df.columns or 'Engagement_Score' not in df.columns:
        return None

    with output:
        print("\n16. Average Engagement Score by Language")
        print("Reveals which language audiences are most engaged.")
        print("Consider localization strategies for high-engagement languages.")

    avg_data = df.groupby('Language')['Engagement_Score'].mean()
    colors = ['#FC4180', '#6056B3', '#FFC554', '#90CA78', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Engagement Score by Language', pad=20, fontsize=14)
    plt.xlabel('Language', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Engagement Score', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_conversion_by_language(df):
    """Plot average conversion rate by language"""
    if 'Language' not in df.columns or 'Conversion_Rate' not in df.columns:
        return None

    with output:
        print("\n17. Average Conversion Rate by Language")
        print("Shows which language audiences convert at the highest rates.")
        print("Prioritize localization efforts for high-converting languages.")

    avg_data = df.groupby('Language')['Conversion_Rate'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average Conversion Rate by Language', pad=20, fontsize=14)
    plt.xlabel('Language', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average Conversion Rate', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2%}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_roi_by_language(df):
    """Plot average ROI by language"""
    if 'Language' not in df.columns or 'ROI' not in df.columns:
        return None

    with output:
        print("\n18. Average ROI by Language")
        print("Identifies which language audiences deliver the best returns.")
        print("Optimize your multilingual marketing strategy based on these insights.")

    avg_data = df.groupby('Language')['ROI'].mean()
    colors = ['#FC4100', '#0056B3', '#FFC55A', '#9BC4F8', '#5905E0']

    plt.figure(figsize=(12, 6))
    bars = avg_data.plot(kind='bar', color=colors)

    plt.title('Average ROI by Language', pad=20, fontsize=14)
    plt.xlabel('Language', labelpad=10)
    plt.xticks(rotation=45)
    plt.ylabel('Average ROI', labelpad=10)

    for bar in bars.patches:
        height = bar.get_height()
        bars.annotate(f'{height:.2f}',
                     (bar.get_x() + bar.get_width() / 2, height),
                     ha='center', va='center',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

def plot_correlation_matrix(df):
    """Plot correlation matrix for key metrics"""
    metrics = ['Engagement_Score', 'Conversion_Rate', 'Acquisition_Cost', 'ROI']
    available_metrics = [m for m in metrics if m in df.columns]

    if len(available_metrics) < 2:
        return None

    with output:
        print("\n19. Correlation Between Key Metrics")
        print("This heatmap reveals relationships between different performance metrics.")
        print("Strong correlations (near 1 or -1) indicate important relationships to explore.")

    correlation_matrix = df[available_metrics].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Between Key Metrics', pad=20, fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_impressions_clicks_ctr(df):
    """Plot impressions, clicks, and CTR by campaign type"""
    if not all(col in df.columns for col in ['Campaign_Type', 'Impressions', 'Clicks']):
        return None

    with output:
        print("\n20. Impressions, Clicks, and CTR by Campaign Type")
        print("Shows reach, engagement, and efficiency of each campaign type.")
        print("High impressions with low CTR may indicate creative fatigue.")

    avg_impressions = df.groupby(['Campaign_Type']).Impressions.mean()
    avg_clicks = df.groupby(['Campaign_Type']).Clicks.mean()
    avg_ctr = avg_clicks / avg_impressions

    bar_width = 0.25
    index = range(len(avg_impressions))

    plt.figure(figsize=(14, 7))
    bars1 = plt.bar(index, avg_impressions, bar_width, label='Impressions', color='#5905E0', alpha=0.7)
    bars2 = plt.bar([i + bar_width for i in index], avg_clicks, bar_width, label='Clicks', color='#FC4100', alpha=0.7)
    bars3 = plt.bar([i + 2 * bar_width for i in index], avg_ctr, bar_width, label='CTR', color='#FFC55A', alpha=0.7)

    plt.title('Average Impressions, Clicks, and CTR by Campaign Type', pad=20, fontsize=14)
    plt.xlabel('Campaign Type', labelpad=10)
    plt.ylabel('Average Value', labelpad=10)
    plt.xticks([i + bar_width for i in index], avg_impressions.index, rotation=45)

    def add_labels(bars, is_ctr=False):
        for bar in bars:
            height = bar.get_height()
            label = f'{height:.2%}' if is_ctr else f'{height:,.0f}'
            plt.text(bar.get_x() + bar.get_width() / 2, height, label,
                    ha='center', va='bottom')

    add_labels(bars1)
    add_labels(bars2)
    add_labels(bars3, is_ctr=True)

    plt.grid(False)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_temporal_performance(df):
    """Plot campaign performance over time"""
    if 'Date' not in df.columns:
        return None

    with output:
        print("\n21. Campaign Performance Over Time")
        print("Tracks how key metrics trend over months and years.")
        print("Identify seasonal patterns or long-term trends in your marketing performance.")

    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month

    metrics = ['Engagement_Score', 'Conversion_Rate', 'ROI']
    available_metrics = [m for m in metrics if m in df.columns]

    if not available_metrics:
        return None

    monthly_performance = df.groupby(['Year', 'Month'])[available_metrics].mean().reset_index()

    for metric in available_metrics:
        plt.figure(figsize=(14, 6))
        lineplot = sns.lineplot(x='Month', y=metric, hue='Year', data=monthly_performance, marker='o')

        for line in lineplot.lines:
            for x_data, y_data in zip(line.get_xdata(), line.get_ydata()):
                plt.text(x_data, y_data,
                        f'{y_data:.2f}' if metric != 'Conversion_Rate' else f'{y_data:.2%}',
                        ha='center', va='bottom', color='#005683')

        plt.title(f'Campaign Performance Over Time - {metric}', pad=20, fontsize=14)
        plt.xlabel('Months', labelpad=10)
        plt.ylabel(metric, labelpad=10)
        plt.xticks(ticks=range(1, 13),
                   labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
        plt.grid(False)
        plt.tight_layout()
        plt.show()

# Function to generate optimization recommendations
def get_recommendations(df):
    """Generate optimization recommendations based on data analysis"""
    recommendations = []

    # 1. Best performing campaign type by conversion rate
    if 'Campaign_Type' in df.columns and 'Conversion_Rate' in df.columns:
        best_conv_type = df.groupby('Campaign_Type')['Conversion_Rate'].mean().idxmax()
        best_conv_rate = df.groupby('Campaign_Type')['Conversion_Rate'].mean().max()
        recommendations.append(
            f"Focus on {best_conv_type} campaigns - they have the highest average conversion rate ({best_conv_rate:.2%})"
        )

    # 2. Best performing channel by ROI
    if 'Channel_Used' in df.columns and 'ROI' in df.columns:
        best_roi_channel = df.groupby('Channel_Used')['ROI'].mean().idxmax()
        best_roi = df.groupby('Channel_Used')['ROI'].mean().max()
        recommendations.append(
            f"Allocate more budget to {best_roi_channel} - it delivers the highest average ROI ({best_roi:.2f})"
        )

    # 3. Best performing customer segment
    if 'Customer_Segment' in df.columns and 'ROI' in df.columns:
        best_segment = df.groupby('Customer_Segment')['ROI'].mean().idxmax()
        best_segment_roi = df.groupby('Customer_Segment')['ROI'].mean().max()
        recommendations.append(
            f"Target more {best_segment} customers - they provide the highest average ROI ({best_segment_roi:.2f})"
        )

    # 4. Best performing duration
    if 'Duration' in df.columns and 'ROI' in df.columns:
        best_duration = df.groupby('Duration')['ROI'].mean().idxmax()
        best_duration_roi = df.groupby('Duration')['ROI'].mean().max()
        recommendations.append(
            f"Consider using {best_duration} campaigns more often - they yield the highest average ROI ({best_duration_roi:.2f})"
        )

    # 5. Check for underperforming campaigns
    if 'ROI' in df.columns and 'Campaign_Type' in df.columns:
        avg_roi = df['ROI'].mean()
        low_roi = df[df['ROI'] < avg_roi * 0.7]  # Campaigns with ROI 30% below average
        if len(low_roi) > 0:
            common_low_types = low_roi['Campaign_Type'].value_counts().index[0]
            recommendations.append(
                f"Review {common_low_types} campaigns - many are performing below average ROI"
            )

    # 6. High cost, low conversion campaigns
    if 'Acquisition_Cost' in df.columns and 'Conversion_Rate' in df.columns and 'Campaign_Type' in df.columns:
        avg_cost = df['Acquisition_Cost'].mean()
        avg_conv = df['Conversion_Rate'].mean()
        high_cost_low_conv = df[(df['Acquisition_Cost'] > avg_cost) & (df['Conversion_Rate'] < avg_conv)]
        if len(high_cost_low_conv) > 0:
            common_types = high_cost_low_conv['Campaign_Type'].value_counts().index[0]
            recommendations.append(
                f"Investigate {common_types} campaigns - some have high costs but below-average conversion rates"
            )

    if not recommendations:
        recommendations.append("Insufficient data for detailed recommendations. Ensure your dataset includes key metrics.")

    return recommendations

# Function to create download link
def create_download_link(df, title="Download results as CSV"):
    """Create a download link for the processed data"""
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"marketing_analysis_results_{timestamp}.csv"
    html = f'<a href="data:file/csv;base64,{b64}" download="{filename}">{title}</a>'
    return HTML(html)

# Event handlers
def on_process_button_clicked(b):
    global df, uploaded_file

    with output:
        clear_output()

        if not upload.value:
            print("Please upload a CSV file first.")
            return

        uploaded_file = next(iter(upload.value.values()))
        df = pd.read_csv(io.BytesIO(uploaded_file['content']))

        # Clean data
        df = clean_data(df)

        print("=== DATA LOADED SUCCESSFULLY ===")
        print("\nData Preview:")
        display(df.head(3))

        print("\nData Information:")
        display(df.info())

        print("\n\n=== PERFORMANCE ANALYSIS ===")

        # Call all visualization functions
        plot_functions = [
            plot_conversion_by_campaign_type,
            plot_roi_by_campaign_type,
            plot_acquisition_cost_by_channel,
            plot_conversion_heatmap,
            plot_engagement_by_segment,
            plot_conversion_by_segment,
            plot_engagement_by_channel,
            plot_conversion_by_channel,
            plot_roi_by_channel,
            plot_engagement_by_duration,
            plot_conversion_by_duration,
            plot_roi_by_duration,
            plot_engagement_by_location,
            plot_conversion_by_location,
            plot_roi_by_location,
            plot_engagement_by_language,
            plot_conversion_by_language,
            plot_roi_by_language,
            plot_correlation_matrix,
            plot_impressions_clicks_ctr,
            plot_temporal_performance
        ]

        for func in plot_functions:
            func(df)

        # Show recommendations
        print("\n=== OPTIMIZATION RECOMMENDATIONS ===")
        recommendations = get_recommendations(df)
        if recommendations:
            for i, rec in enumerate(recommendations, 1):
                print(f"{i}. {rec}")
        else:
            print("Could not generate specific recommendations. Please ensure your data includes:")
            print("- Campaign_Type, Channel_Used, Target_Audience")
            print("- Conversion_Rate, ROI, Acquisition_Cost")
            print("- Engagement_Score, Duration, Location")

        # Add download link
        print("\n\n=== EXPORT RESULTS ===")
        display(create_download_link(df))

        print("\nAnalysis complete! Scroll up to review all insights.")

def on_open_window_clicked(b):
    open_dashboard()

# Assign event handlers
process_button.on_click(on_process_button_clicked)
open_window_button.on_click(on_open_window_clicked)

<IPython.core.display.Javascript object>